# 宽长转换与透视汇总

学习目标：在宽表、长表和多级索引之间重排数据，按明确口径透视汇总，并检查重复键、缺失组合与展开后的行数。

前置知识：DataFrame、分组聚合、索引与元组标签、分类类型、缺失值。

运行环境：Python 3.12、pandas 3.0；示例按 pandas 3.0.6 编写。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用自制小表，后续单元沿用已导入的 pd。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 把测量列整理成长表

宽表用不同列保存同类测量，长表把测量名称放到一列、测量值放到另一列。melt 将指定的测量列展开，同时重复保留标识列，便于按测量名称筛选和汇总。

id_vars 指定标识列，value_vars 指定展开的列，var_name 和 value_name 给新列命名。下面把早晚温度整理成“站点、时段、温度”三列，温度单位为摄氏度。

In [1]:
import pandas as pd

wide = pd.DataFrame({"station": ["A", "B"], "morning": [18, 20], "evening": [21, 23]})
long = wide.melt(id_vars="station", value_vars=["morning", "evening"],
                 var_name="period", value_name="temperature_c")
print(long)  # 先 A、B 的 morning，再 A、B 的 evening；测量值为 18、20、21、23。
print(long.shape, long.dtypes)  # (4, 3)，标识与时段为 str，温度为 int64。
print(long.groupby("period", sort=False)["temperature_c"].mean())  # morning 19，evening 22。

  station   period  temperature_c
0       A  morning             18
1       B  morning             20
2       A  evening             21
3       B  evening             23
(4, 3) station            str
period             str
temperature_c    int64
dtype: object
period
morning    19.0
evening    22.0
Name: temperature_c, dtype: float64


## 2 保留标识与恢复宽表

### 2.1 行标签是否保留

melt 默认 ignore_index=True，重新生成行索引。设为 False 会重复原行标签；这能保存来源位置，但不保证结果索引唯一。真正的业务标识最好明确放在 id_vars 中。

下面沿用 wide，仅为原表增加便于辨认的行标签。

In [2]:
labeled = wide.set_axis(["R1", "R2"])
kept = labeled.melt(id_vars="station", value_vars=["morning", "evening"],
                    var_name="period", value_name="temperature_c", ignore_index=False)
print(kept.index.tolist(), kept.index.is_unique)  # ['R1', 'R2', 'R1', 'R2'] False。
print(kept["station"].tolist())  # ['A', 'B', 'A', 'B']，标识列也被保留。

['R1', 'R2', 'R1', 'R2'] False
['A', 'B', 'A', 'B']


### 2.2 pivot 重排唯一组合

pivot 指定哪些列的值成为行标签、列标签与单元格。它只重排，不聚合；每个“行键、列键”组合必须唯一。

两幅官方图用相同颜色连接宽表中的列与长表中的记录：标识随每个测量字段重复，字段名称变成长表的一列，测量值仍与原标识对应。

![pandas 官方 melt 图：将宽表的测量列展开成长表记录。](image/illustration/13-01-wide-long.svg)

![pandas 官方 pivot 图：按标识与字段将长表重排为宽表。](image/illustration/13-02-long-wide.svg)

引用 pandas 官方原图，保留原色块与内容；两图表示布局转换，没有标注本例的站点、时段或温度。版权与 BSD-3-Clause 许可见篇末。

继续使用 wide 与 long，追踪 A 的 evening=21：宽表列名变成长表 period 的取值，温度由 station 与 period 共同定位。下面 pivot 恢复宽表时，显式恢复 morning、evening 的列顺序，再核对轴名称与 dtype；不能只因数字相同就认为往返完全一致。

In [3]:
restored = long.pivot(index="station", columns="period", values="temperature_c")
restored = restored.reindex(columns=["morning", "evening"])
expected = wide.set_index("station")
restored = restored.rename_axis(columns=None)
print(restored)  # A 行为 18、21，B 行为 20、23，形状 (2, 2)。
print(restored.dtypes)  # 两列仍为 int64。
pd.testing.assert_frame_equal(restored, expected)
print("值、标签、顺序、轴名称和类型一致")  # 预期：往返转换的各项断言通过后显示此提示。

         morning  evening
station                  
A             18       21
B             20       23
morning    int64
evening    int64
dtype: object
值、标签、顺序、轴名称和类型一致


## 3 重复键与聚合选择

### 3.1 pivot 不替你决定口径

同一个站点、时段存在多条测量时，一个单元格无法原样放入多个标量。pivot 会拒绝重复组合；即使重复记录的值相同，也需要先决定保留、去重或聚合规则。

下面用独立的销售数量表演示。同一地区、商品的两行代表两笔销售，应该相加，不能任意删除。

In [4]:
sales = pd.DataFrame({"region": ["east", "east", "west"],
                      "item": ["A", "A", "B"], "quantity": [2, 4, 3]})
print(sales.duplicated(["region", "item"], keep=False).tolist())  # [True, True, False]。

# 预期 ValueError：east/A 对应两条记录，pivot 无法把两个值放进同一个单元格。
sales.pivot(index="region", columns="item", values="quantity")

[True, True, False]


ValueError: Index contains duplicate entries, cannot reshape

### 3.2 pivot_table 指定聚合

pivot_table 可以对同一组合聚合，默认 aggfunc="mean"。销售件数需要合计时应显式写 sum；不同口径不能混用。fill_value 在聚合完成后填补结果中的缺失，不代表原始输入本来就是零。

下面约定没有销售组合时展示 0。聚合压缩了明细信息，汇总表无法恢复每笔销售。

In [5]:
totals = pd.pivot_table(sales, index="region", columns="item", values="quantity",
                        aggfunc="sum", fill_value=0, observed=True, sort=True)
print(totals)  # east 的 A 为 6，west 的 B 为 3，另两个组合展示 0。
print(totals.shape, totals.dtypes.tolist())  # (2, 2)，两列 int64。
print(totals.to_numpy().sum(), sales["quantity"].sum())  # 都是 9，合计未丢失。
assert totals.to_numpy().sum() == sales["quantity"].sum()
means = pd.pivot_table(sales, index="region", columns="item", values="quantity", aggfunc="mean")
print(means.loc["east", "A"])  # 3.0，平均每笔 3 件，不是合计 6 件。

item    A  B
region      
east    6  0
west    0  3
(2, 2) [dtype('int64'), dtype('int64')]
9 9
3.0


## 4 分类、缺失组合与总计

### 4.1 observed 与 dropna

分类分组中的 observed 决定是否只保留实际观测类别，pandas 3 默认 True。需要展示预设的未出现类别时，显式设为 False，并检查 dropna 对全缺失列和缺失分组键的影响。

下面将 sales 中的地区和商品转为固定类别，用均值展示未观测组合。没有观测与观测后得到零是不同状态，不应一概填零。

In [6]:
categorized = sales.copy()
categorized["region"] = categorized["region"].astype(pd.CategoricalDtype(["east", "west", "north"]))
categorized["item"] = categorized["item"].astype(pd.CategoricalDtype(["A", "B", "C"]))
observed = pd.pivot_table(categorized, index="region", columns="item", values="quantity",
                          aggfunc="mean", observed=True, dropna=False)
all_categories = pd.pivot_table(categorized, index="region", columns="item", values="quantity",
                                aggfunc="mean", observed=False, dropna=False)
print(observed.shape, all_categories.shape)  # (2, 2) 与 (3, 3)。
print(all_categories)  # east-A、west-B 为 3；north 行和 C 列均为 NaN。

(2, 2) (3, 3)
item      A    B   C
region              
east    3.0  NaN NaN
west    NaN  3.0 NaN
north   NaN  NaN NaN


### 4.2 margins 使用同一聚合口径

margins=True 增加总计行和列，margins_name 修改其名称。它使用 aggfunc 对相应原始数据聚合；均值总计不能简单地把各组均值相加或不加权平均。

dropna=True 时，含缺失值的行会影响总计所使用的记录集合。下面的 sales 无缺失，先查看合计，再把 west 那笔改为 9 件，观察总均值如何按三笔原始记录计算。

In [7]:
with_total = pd.pivot_table(sales, index="region", columns="item", values="quantity",
                            aggfunc="sum", fill_value=0, margins=True, margins_name="Total",
                            observed=True, dropna=True)
print(with_total)  # east 合计 6，west 合计 3，右下总计为 9。
changed_sales = sales.assign(quantity=[2, 4, 9])
average = pd.pivot_table(changed_sales, index="region", values="quantity", aggfunc="mean",
                         margins=True, margins_name="Total")
print(average)  # east 均值 3，west 均值 9；总均值为 5，而非组均值不加权平均得到的 6。
print(with_total.loc["Total", "Total"] == sales["quantity"].sum())  # True。

item    A  B  Total
region             
east    6  0      6
west    0  3      3
Total   6  3      9
        quantity
region          
east         3.0
west         9.0
Total        5.0
True


### 4.3 缺失键与缺失测量分开检查

缺失分组键意味着记录无法归入普通标签；缺失测量意味着某项数值未知。dropna=False 可以保留缺失键，但聚合本身仍有缺失处理规则。

下面使用均值，不额外填零。检查有效值数量有助于解释结果。

In [8]:
incomplete = pd.DataFrame({"region": ["east", None, "west"],
                           "item": ["A", "A", "B"], "quantity": [2.0, 5.0, None]})
kept_keys = pd.pivot_table(incomplete, index="region", columns="item", values="quantity",
                          aggfunc="mean", dropna=False, observed=True)
print(kept_keys)  # 缺失地区的 A 为 5；west-B 缺失，不能当作零销售。
print(incomplete["quantity"].count(), incomplete["region"].isna().sum())  # 2 个有效值，1 个缺失键。

item      A   B
region         
east    2.0 NaN
west    NaN NaN
NaN     5.0 NaN
2 1


## 5 在列与索引层级之间移动

### 5.1 stack 保留原始缺失

stack 把列标签移入行索引的内层。单层列转换后得到 Series，其 MultiIndex 可用“原行标签、原列标签”的元组定位；unstack 则把指定索引层级移回列。

pandas 3 默认使用新的 stack 实现，保留输入的缺失且不自动排序；不要再传 dropna 或 sort。需要删除缺失或排序时，另做显式操作。

In [9]:
measurements = pd.DataFrame({"morning": [18.0, None], "evening": [21.0, 23.0]},
                            index=pd.Index(["A", "B"], name="station"))
measurements.columns.name = "period"
stacked = measurements.stack()
print(stacked)  # A-morning、A-evening、B-morning、B-evening；B-morning 的 NaN 仍在。
print(stacked.index.names, len(stacked))  # ['station', 'period']，4。
print(stacked.loc[("A", "evening")])  # 21.0，元组指定两个层级。
print(len(stacked.dropna()))  # 3，删除是后续的显式决定。

station  period 
A        morning    18.0
         evening    21.0
B        morning     NaN
         evening    23.0
dtype: float64
['station', 'period'] 4
21.0
3


### 5.2 unstack 与新增缺失

unstack 的 level 可以使用层级名称，默认移出最后一层。缺失的键组合会产生空单元格；fill_value 可以填充这种重排新增的位置，但不替换原来已有的缺失值。

下面先恢复 measurements，再删除其中一个组合，对照“没有这条组合”与“组合存在但测量缺失”。

In [10]:
back = stacked.unstack("period").reindex(columns=measurements.columns)
pd.testing.assert_frame_equal(back, measurements)
print(back.equals(measurements))  # True，恢复了标签、类型及缺失位置。
partial = stacked.drop(index=("A", "evening"))
filled = partial.unstack("period", fill_value=0)
print(filled)  # A-evening 新增位置填 0；B-morning 原始 NaN 保留。
print(filled.loc["A", "evening"], pd.isna(filled.loc["B", "morning"]))  # 0.0 True。

True
period   morning  evening
station                  
A           18.0      0.0
B            NaN     23.0
0.0 True


## 6 交叉表

crosstab 默认统计两个分类变量组合的频数；它统计记录条数，不等于某个数量列求和。normalize="index" 按行归一化，"columns" 按列归一化，"all" 按全表归一化。

下面继续用 sales，观察地区内各商品的记录比例；要汇总件数，需要同时指定 values 与 aggfunc。

In [11]:
frequency = pd.crosstab(sales["region"], sales["item"], margins=True)
proportions = pd.crosstab(sales["region"], sales["item"], normalize="index")
print(frequency)  # east-A 有 2 笔，west-B 有 1 笔；All-All 为 3。
print(proportions)  # east 的 A 比例为 1，west 的 B 比例为 1。
print(pd.crosstab(sales["region"], sales["item"], values=sales["quantity"], aggfunc="sum"))
# 这里统计件数，east-A 为 6、west-B 为 3；未出现组合为 NaN。

item    A  B  All
region           
east    2  0    2
west    0  1    1
All     2  1    3


item      A    B
region          
east    1.0  0.0
west    0.0  1.0
item      A    B
region          
east    6.0  NaN
west    NaN  3.0


### 6.1 保留未出现的类别

crosstab 接收分类数据时，可以用 dropna=False 保留未出现的类别。下面沿用 categorized，明确“零条记录”的计数含义；零频数不能推断其他数值指标也是零。

In [12]:
full_frequency = pd.crosstab(categorized["region"], categorized["item"], dropna=False)
print(full_frequency)  # north 行、C 列均为 0，完整形状为 (3, 3)。
print(full_frequency.shape, full_frequency.to_numpy().sum())  # (3, 3) 3。

item    A  B  C
region         
east    2  0  0
west    0  1  0
north   0  0  0
(3, 3) 3


## 7 展开列表列

explode 把一行中的列表展开成多行，其他列随之重复。空列表会保留一行缺失，标量保持一行；默认重复原行索引，ignore_index=True 则重新编号。

集合展开的顺序不确定；需要稳定顺序时使用有序列表。展开后应重新检查 dtype，不依赖旧版本“总是 object”的概括。

In [13]:
tagged = pd.DataFrame({"id": ["A", "B", "C"], "tags": [["hot", "new"], [], ["sale"]]})
expanded = tagged.explode("tags")
print(expanded)  # A 两行、B 一行缺失、C 一行，共 4 行。
print(expanded.index.tolist(), expanded.shape)  # [0, 0, 1, 2] (4, 2)。
print(expanded.dtypes)  # 当前环境两列为 str，tags 原输入为装有列表的 object。
print(expanded["tags"].notna().sum())  # 3 个有效标签，不是 4 个。

  id  tags
0  A   hot
0  A   new
1  B   NaN
2  C  sale
[0, 0, 1, 2] (4, 2)
id      str
tags    str
dtype: object
3


## 8 选学：多组字段与同步展开

### 8.1 wide_to_long 识别列名模式

wide_to_long 适合“共同前缀 + 分隔符 + 后缀”的列名。stubnames 指定前缀，i 指定唯一标识列，j 给后缀对应的索引层命名，sep 指定分隔符。默认 suffix 匹配数字后缀。

下面把两年的销量与成本同步展开；成本单位为元，输入用整数表示本例金额。

In [14]:
annual = pd.DataFrame({"shop": ["A", "B"], "sales_2024": [10, 20], "sales_2025": [12, 25],
                       "cost_2024": [5, 8], "cost_2025": [6, 9]})
annual_long = pd.wide_to_long(annual, stubnames=["sales", "cost"], i="shop", j="year", sep="_")
print(annual_long.sort_index())  # A 的两年销售为 10、12；B 为 20、25，成本也按年份对应。
print(annual_long.index.names, annual_long.shape)  # ['shop', 'year'] (4, 2)。

           sales  cost
shop year             
A    2024     10     5
     2025     12     6
B    2024     20     8
     2025     25     9


['shop', 'year'] (4, 2)


### 8.2 多列 explode 要求逐行等长

一次展开多个列时，同一行各列表长度必须一致，表示对应位置的数据成对出现。先展开一列再展开另一列会产生不同的组合关系，不能当作同步展开。

下面的商品和件数按位置一一对应；不一致的行应先作为输入错误处理。

In [15]:
bundles = pd.DataFrame({"order": ["O1", "O2"], "item": [["A", "B"], ["C"]],
                        "quantity": [[2, 3], [4]]})
details = bundles.explode(["item", "quantity"], ignore_index=True)
print(details)  # O1 的 A 对应 2、B 对应 3；O2 的 C 对应 4，共 3 行。
print(details.shape, details.dtypes)  # 展开后重新查看类型，再按业务需要转换。
invalid = pd.DataFrame({"item": [["A", "B"]], "quantity": [[2]]})

# 预期 ValueError：同一行的 item 有两个元素、quantity 只有一个，无法同步展开。
invalid.explode(["item", "quantity"])

  order item quantity
0    O1    A        2
1    O1    B        3
2    O2    C        4
(3, 3) order          str
item           str
quantity    object
dtype: object


ValueError: columns must have matching element counts

## 本章小结

（1）melt 把测量列展开，pivot 把唯一键组合重排；检查标识、顺序、类型与轴名称才能判断往返是否一致。

（2）pivot_table 和 crosstab 按指定口径聚合，可能压缩明细。频数、合计、均值与归一化比例分别回答不同问题。

（3）分类参数、缺失键、未出现组合与总计都会影响输出；展示为零必须有明确含义。

（4）stack、unstack 移动轴层级；explode 展开列表。重排后核对缺失位置、索引和记录数量。

## 练习

（1）先预测以下两次展开后的行数、标签和缺失位置，再运行。为什么两行输入不意味着展开后仍是两行？

In [16]:
example = pd.DataFrame({"id": ["A", "B"], "first": [1, 2], "second": [3, 4]})
print(example.melt(id_vars="id", ignore_index=False))
lists = pd.DataFrame({"id": ["A", "B"], "values": [[1, 2], []]})
print(lists.explode("values"))
# 运行前记录预测；区分重复标识、重复索引与空列表保留行。

  id variable  value
0  A    first      1
1  B    first      2
0  A   second      3
1  B   second      4
  id values
0  A      1
0  A      2
1  B    NaN


（2）把下面的测量表转成长表，再恢复宽表。保留站点标识和原列顺序，检查值、dtype、轴名称及缺失位置，不把缺失填成 0。

In [17]:
readings = pd.DataFrame({"station": ["B", "A"], "first": [1.0, None], "second": [3.0, 4.0]})
# 在此 melt、pivot，再显式恢复 B、A 的行顺序及 first、second 的列顺序。
# 使用 assert_frame_equal 对照 readings.set_index('station')，先统一轴名称。

（3）原任务要求保留每笔销售，现在改成按地区和商品统计总件数。选择 pivot 或 pivot_table 并解释理由；另外生成记录频数表，说明为什么它不等于总件数表。

In [18]:
transactions = pd.DataFrame({"region": ["east", "east", "west"],
                             "item": ["A", "A", "A"], "quantity": [2, 5, 4]})
# 在此汇总并生成频数表。
# 检查：east-A 总件数 7、记录数 2；west-A 总件数 4、记录数 1。
# 说明聚合后为何不能恢复 east 的两笔原始件数。

（4）有一列列表标签需要展开，但报告同时要求“原记录数量”和“有效标签数量”。完成展开并分别统计；解释空列表对应的缺失行为何不算有效标签。

In [19]:
records = pd.DataFrame({"id": ["A", "B", "C"], "tags": [["x", "y"], [], ["z", "w"]]})
# 在此展开，保留 id；检查原记录为 3 条、展开后 5 行、有效标签为 4 个。
# 不能把展开后行数当作原记录数，也不能把空列表产生的行计为一个标签。

### 重点练习提示（第 3 题）

提示一：地区与商品的组合不唯一，先判断是否允许把多笔记录压成一项。

提示二：总件数需要对 quantity 求和；频数表只计记录条数。

### 参考解析（第 3 题）

使用 pivot_table，index="region"、columns="item"、values="quantity"、aggfunc="sum"，并显式给出 sort、observed、dropna。本题得到 east/A=7、west/A=4；pd.crosstab(transactions["region"], transactions["item"]) 则得到 2、1。pivot 无法接受重复的 east/A 组合。求和后只保留 7，无法恢复原先的 2 与 5，因此若仍需要每笔销售，应保留 transactions；不能用聚合后的表冒充明细。

## 参考与引用来源

本章机制示意图由 CMYK Labs 根据所列第一方资料与教学输入绘制；图用于解释关系，不作为实际运行截图。

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [melt](https://pandas.pydata.org/docs/reference/api/pandas.melt.html) 的 id_vars、value_vars、ignore_index、var_name、value_name；[DataFrame.pivot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html) 的 Parameters 与重复键 Raises；[pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html) 的 aggfunc、fill_value、margins、dropna、observed（3.0 默认 True）与 sort；[DataFrame.stack](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html) 的默认 future_stack=True、参数约束和输出类型；[2.1.0 新 stack 实现说明](https://pandas.pydata.org/docs/whatsnew/v2.1.0.html#new-implementation-of-dataframe-stack) 的缺失保留与顺序，对照当前接口；[DataFrame.unstack](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.unstack.html) 的 level、fill_value 与排序；[crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) 的频数、values/aggfunc、normalize、margins 和保留未观测类别示例；[DataFrame.explode](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.explode.html) 的多列等长条件、ignore_index、空列表、集合顺序与 dtype（以本例保存输出注明当前类型）；[wide_to_long](https://pandas.pydata.org/docs/reference/api/pandas.wide_to_long.html) 的 stubnames、i、j、sep、suffix。  图源（官方文档 3.0.6，2026-09-22 核查）：[How to reshape the layout of tables](https://pandas.pydata.org/docs/getting_started/intro_tutorials/07_reshape_table_layout.html) 的 Long to wide table format、Wide to long format；原图 [07_melt.svg](https://pandas.pydata.org/docs/_images/07_melt.svg)、[07_pivot.svg](https://pandas.pydata.org/docs/_images/07_pivot.svg)。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[v2.1.0](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/whatsnew/v2.1.0.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。  图源版权与许可：[pandas v3.0.6 LICENSE](https://github.com/pandas-dev/pandas/blob/v3.0.6/LICENSE)，完整 BSD-3-Clause 条款。 |